# 1、导入相关的库


In [ ]:
import os
import tqdm
import torch
import pickle
import tarfile
import logging
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import List, Optional, Union, Dict, Any
from torch.utils.data import DataLoader, TensorDataset

# 2、环境配置


In [ ]:
# 设置随机数种子
RANDOM_SEED = 2025

# 配置日志记录器
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)  # 仅控制台输出
logger = logging.getLogger("Record")


# 设置随机种子
def set_seed(seed=RANDOM_SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()

# 3、数据加载和数据预处理


In [ ]:
class CIFAR10Loader:
    """
    CIFAR-10数据加载器，支持数据加载、预处理、可视化和统计信息显示
    """

    def __init__(self, data_path="data/cifar-10-python.tar.gz", val_split=0.1):
        """
        初始化CIFAR-10数据加载器

        Args:
            data_path (str): CIFAR-10数据集路径
            val_split (float): 验证集比例 (0.0-1.0)
        """

        self.data_path = data_path
        self.val_split = val_split
        self.class_names = [
            "airplane",
            "automobile",
            "bird",
            "cat",
            "deer",
            "dog",
            "frog",
            "horse",
            "ship",
            "truck",
        ]

        # 存储加载的数据
        self.train_data = None
        self.train_labels = None
        self.val_data = None
        self.val_labels = None
        self.test_data = None
        self.test_labels = None

        # 原始数据
        self.original_train_data = None
        self.original_train_labels = None
        self.original_test_data = None
        self.original_test_labels = None

    def _read_cifar10_batch(self, batch_file):
        """
        读取CIFAR-10单个批次文件
        """

        with open(batch_file, "rb") as file:
            batch_data = pickle.load(file, encoding="bytes")

        data = batch_data[b"data"]
        labels = np.array(batch_data[b"labels"])

        data = data.reshape(len(data), 3, 32, 32).transpose(0, 2, 3, 1)

        return data, labels

    def _read_cifar10_train_set(self, batch_files):
        """
        读取所有CIFAR-10训练批次
        """

        all_data = []
        all_labels = []

        for batch_file in batch_files:
            data, labels = self._read_cifar10_batch(batch_file)
            all_data.append(data)
            all_labels.append(labels)

        # 合并所有批次
        train_data = np.vstack(all_data)
        train_labels = np.hstack(all_labels)

        return train_data, train_labels

    def _read_cifar10_meta(self, meta_file):
        """
        读取CIFAR-10元数据
        """

        try:
            with open(meta_file, "rb") as file:
                meta_data = pickle.load(file, encoding="bytes")
            class_names = [name.decode("utf-8") for name in meta_data[b"label_names"]]
            return class_names
        except FileNotFoundError:
            logger.warning(
                f"Meta file {meta_file} not found, using default class names"
            )
            return self.class_names

    def extract_and_load_data(self):
        """
        提取并加载CIFAR-10数据
        """

        if self.data_path.endswith(".tar.gz"):
            extract_path = os.path.dirname(self.data_path)
            cifar_dir = os.path.join(extract_path, "cifar-10-batches-py")
            if not os.path.exists(cifar_dir):
                logger.info("Extracting CIFAR-10 dataset...")
                with tarfile.open(self.data_path, "r:gz") as tar:
                    tar.extractall(extract_path, filter="data")
                logger.info("Extraction completed")
            else:
                logger.info("CIFAR-10数据已存在，跳过解压")
        else:
            cifar_dir = self.data_path

        logger.info("Loading CIFAR-10 train set...")

        # 构建批次文件路径
        train_batch_files = [
            os.path.join(cifar_dir, f"data_batch_{i}") for i in range(1, 6)
        ]
        test_batch_file = os.path.join(cifar_dir, "test_batch")
        meta_file = os.path.join(cifar_dir, "batches.meta")

        # 加载训练数据
        train_data, train_labels = self._read_cifar10_train_set(train_batch_files)

        # 保存原始数据（用于可视化）
        self.original_train_data = train_data.copy()
        self.original_train_labels = train_labels.copy()

        # 数据预处理：归一化到[0,1]范围
        train_data = train_data.astype(np.float32) / 255.0

        # 使用train_test_split划分训练集和验证集（分层抽样）
        train_data, val_data, train_labels, val_labels = train_test_split(
            train_data,
            train_labels,
            test_size=self.val_split,
            random_state=RANDOM_SEED,
            stratify=train_labels,  # 保持类别分布一致
        )

        logger.info(f"train_data:   [{str(train_data.dtype)}] {train_data.shape}")
        logger.info(f"train_labels: [{str(train_labels.dtype)}] {train_labels.shape}")
        logger.info(f"val_data:     [{str(val_data.dtype)}] {val_data.shape}")
        logger.info(f"val_labels:   [{str(val_labels.dtype)}] {val_labels.shape}")

        logger.info("Loading CIFAR-10 test set...")

        # 加载测试数据
        test_data, test_labels = self._read_cifar10_batch(test_batch_file)

        # 保存原始测试数据
        self.original_test_data = test_data.copy()
        self.original_test_labels = test_labels.copy()

        test_data = test_data.astype(np.float32) / 255.0

        logger.info(f"test_data:   [{str(test_data.dtype)}] {test_data.shape}")
        logger.info(f"test_labels: [{str(test_labels.dtype)}] {test_labels.shape}")

        # 加载类别名称
        self.class_names = self._read_cifar10_meta(meta_file)
        logger.info(f"CIFAR-10 Classes: {self.class_names}")

        # 存储数据
        self.train_data = train_data
        self.train_labels = train_labels
        self.val_data = val_data
        self.val_labels = val_labels
        self.test_data = test_data
        self.test_labels = test_labels

        return (
            (train_data, train_labels),
            (val_data, val_labels),
            (test_data, test_labels),
        )

    def preprocess_data_for_pytorch(self, standardize=True):
        """
        为PyTorch预处理数据

        Args:
            standardize (bool): 是否进行标准化

        Returns:
            tuple: ((训练数据, 训练标签), (验证数据, 验证标签), (测试数据, 测试标签))
        """

        if self.train_data is None:
            raise ValueError("请先调用 extract_and_load_data() 方法加载数据")

        # 重塑数据：从(N, 32, 32, 3)到(N, 3, 32, 32)
        train_data = self.train_data.transpose(0, 3, 1, 2)
        val_data = self.val_data.transpose(0, 3, 1, 2)
        test_data = self.test_data.transpose(0, 3, 1, 2)

        if standardize:
            # 标准化（零均值，单位方差）
            mean = train_data.mean(axis=(0, 2, 3), keepdims=True)
            std = train_data.std(axis=(0, 2, 3), keepdims=True)
            train_data = (train_data - mean) / (std + 1e-8)
            val_data = (val_data - mean) / (std + 1e-8)
            test_data = (test_data - mean) / (std + 1e-8)

        # 转换为PyTorch张量
        train_data = torch.FloatTensor(train_data)
        val_data = torch.FloatTensor(val_data)
        test_data = torch.FloatTensor(test_data)
        train_labels = torch.LongTensor(self.train_labels)
        val_labels = torch.LongTensor(self.val_labels)
        test_labels = torch.LongTensor(self.test_labels)

        return (
            (train_data, train_labels),
            (val_data, val_labels),
            (test_data, test_labels),
        )

    def visualize_dataset(self, samples_per_class=10, figsize=(16, 12), dpi=300):
        """
        可视化数据集样本
        """

        if self.original_train_data is None:
            raise ValueError("请先调用 extract_and_load_data() 方法加载数据")

        fig, axes = plt.subplots(10, samples_per_class, figsize=figsize, dpi=dpi)
        fig.suptitle("CIFAR-10 Dataset Preview", fontsize=16, x=0.57, y=0.98)

        for label in range(10):
            # 找到该类别的前N个样本
            class_indices = np.where(self.original_train_labels == label)[0][
                :samples_per_class
            ]

            for i, img_index in enumerate(class_indices):
                ax = axes[label, i]
                ax.imshow(self.original_train_data[img_index])
                ax.axis("off")

            # 在每一行的左侧添加类别标签
            axes[label, 0].text(
                -0.1,
                0.5,
                f"{label}: {self.class_names[label]}",
                transform=axes[label, 0].transAxes,
                fontsize=10,
                verticalalignment="center",
                horizontalalignment="right",
                rotation=0,
            )

        plt.tight_layout()
        plt.subplots_adjust(left=0.15, top=0.94)
        plt.show()

    def show_statistics(self):
        """
        显示数据集统计信息
        """

        if self.train_data is None:
            raise ValueError("请先调用 extract_and_load_data() 方法加载数据")

        print("\nDataset Statistics:")
        print(f"Image shape: {self.train_data.shape[1:]}")
        print(f"Number of classes: {len(self.class_names)}")
        print(f"Training samples: {len(self.train_data)}")
        print(f"Validation samples: {len(self.val_data)}")
        print(f"Test samples: {len(self.test_data)}")

        # 每个类别的样本数量统计
        print("\nClass distribution in training set:")
        unique, counts = np.unique(self.train_labels, return_counts=True)
        for class_id, count in zip(unique, counts):
            print(f"  {self.class_names[class_id]:>10}: {count:>5} samples")

        # 验证集类别分布
        print("\nClass distribution in validation set:")
        unique_val, counts_val = np.unique(self.val_labels, return_counts=True)
        for class_id, count in zip(unique_val, counts_val):
            print(f"  {self.class_names[class_id]:>10}: {count:>5} samples")

    def get_data_loaders(self, batch_size=64, num_workers=0, shuffle=True):
        """
        获取PyTorch数据加载器
        """

        (train_data, train_labels), (val_data, val_labels), (test_data, test_labels) = (
            self.preprocess_data_for_pytorch()
        )

        # 创建数据集
        train_dataset = TensorDataset(train_data, train_labels)
        val_dataset = TensorDataset(val_data, val_labels)
        test_dataset = TensorDataset(test_data, test_labels)

        # 创建数据加载器
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            worker_init_fn=np.random.seed(torch.initial_seed() % 2**32),
            generator=torch.Generator().manual_seed(RANDOM_SEED),
        )
        val_loader = DataLoader(
            val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
        )
        test_loader = DataLoader(
            test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
        )

        return train_loader, val_loader, test_loader

# 4、模型构建


## 4.1、多层感知机模型


In [ ]:
class MLP(nn.Module):
    """
    多层感知机模型
    """

    def __init__(
        self,
        input_size=3072,
        hidden_sizes=[512, 256, 128],
        num_classes=10,
        dropouts=[0.1, 0.2, 0.3, 0.4],
        use_batch_norm=True,
    ):
        super(MLP, self).__init__()

        self.use_batch_norm = use_batch_norm
        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropouts])
        self.batch_norms = nn.ModuleList() if use_batch_norm else None

        self.layers.append(nn.Linear(input_size, hidden_sizes[0]))
        if use_batch_norm:
            self.batch_norms.append(nn.BatchNorm1d(hidden_sizes[0]))

        for i in range(len(hidden_sizes) - 1):
            self.layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))
            if use_batch_norm:
                self.batch_norms.append(nn.BatchNorm1d(hidden_sizes[i + 1]))

        self.layers.append(nn.Linear(hidden_sizes[-1], num_classes))
        self.activation = nn.ReLU()

        self._initialize_weights()

    def _initialize_weights(self):
        """
        权重初始化
        """

        for layer in self.layers[:-1]:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(
                    layer.weight, mode="fan_out", nonlinearity="relu"
                )
                nn.init.constant_(layer.bias, 0)
        nn.init.normal_(self.layers[-1].weight, std=0.01)
        nn.init.constant_(self.layers[-1].bias, 0)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            if self.use_batch_norm and i < len(self.batch_norms):
                x = self.batch_norms[i](x)
            x = self.activation(x)
            x = self.dropouts[i](x)

        x = self.layers[-1](x)
        return x

## 4.2、卷积神经网路模型


In [ ]:
class ConvNet(nn.Module):
    """
    卷积神经网络，支持多种超参数配置
    """

    def __init__(
        self,
        input_channels: int = 3,
        input_size: int = 32,
        num_classes: int = 10,
        conv_layers_config: Optional[List[Dict[str, Any]]] = None,
        fc_layers_config: Optional[List[int]] = None,
        activation: str = "relu",
        use_batch_norm: bool = True,
        dropout_rates: Union[float, List[float]] = 0.2,
        global_pool: str = "adaptive_avg",  # 'adaptive_avg', 'adaptive_max', 'avg', 'max'
        init_weights: bool = True,
    ):
        """
        Args:
            input_channels: 输入图像通道数
            input_size: 输入图像尺寸（正方形）
            num_classes: 分类类别数
            conv_layers_config: 卷积层配置列表
            fc_layers_config: 全连接层配置列表
            activation: 激活函数类型
            use_batch_norm: 是否使用BatchNorm
            dropout_rates: Dropout率
            global_pool: 全局池化方式
            init_weights: 是否初始化权重
        """

        super(ConvNet, self).__init__()

        self.input_channels = input_channels
        self.input_size = input_size
        self.num_classes = num_classes
        self.activation = self._get_activation_fn(activation)
        self.use_batch_norm = use_batch_norm
        self.global_pool = global_pool

        if conv_layers_config is None:
            conv_layers_config = [
                {
                    "out_channels": 32,
                    "num_conv_layers": 2,
                    "kernel_size": 3,
                    "pool_size": 2,
                },
                {
                    "out_channels": 64,
                    "num_conv_layers": 2,
                    "kernel_size": 3,
                    "pool_size": 2,
                },
                {
                    "out_channels": 128,
                    "num_conv_layers": 2,
                    "kernel_size": 3,
                    "pool_size": 2,
                },
                {
                    "out_channels": 256,
                    "num_conv_layers": 1,
                    "kernel_size": 3,
                    "pool_size": 2,
                },
            ]

        if fc_layers_config is None:
            fc_layers_config = [128]

        if isinstance(dropout_rates, float):
            self.dropout_rates = [dropout_rates] * (
                len(conv_layers_config) + len(fc_layers_config)
            )
        else:
            self.dropout_rates = dropout_rates + [0.5] * max(
                0, len(conv_layers_config) + len(fc_layers_config) - len(dropout_rates)
            )

        self.conv_blocks = nn.ModuleList()
        in_channels = input_channels
        current_size = input_size

        for i, config in enumerate(conv_layers_config):
            conv_block, out_channels, current_size = self._make_conv_block(
                in_channels=in_channels,
                out_channels=config["out_channels"],
                num_conv_layers=config["num_conv_layers"],
                kernel_size=config.get("kernel_size", 3),
                pool_size=config.get("pool_size", 2),
                dropout_rate=self.dropout_rates[i],
                current_size=current_size,
            )
            self.conv_blocks.append(conv_block)
            in_channels = out_channels

        self.feature_size = self._calculate_feature_size(in_channels, current_size)

        self.fc_layers = self._make_fc_layers(
            input_size=self.feature_size,
            hidden_layers=fc_layers_config,
            output_size=num_classes,
            dropout_rates=self.dropout_rates[len(conv_layers_config) :],
        )

        if init_weights:
            self._initialize_weights()

    def _get_activation_fn(self, activation: str) -> nn.Module:
        """
        获取激活函数
        """

        activations = {
            "relu": nn.ReLU(inplace=True),
            "leaky_relu": nn.LeakyReLU(0.1, inplace=True),
            "elu": nn.ELU(inplace=True),
            "gelu": nn.GELU(),
            "swish": nn.SiLU(),
            "mish": nn.Mish() if hasattr(nn, "Mish") else nn.ReLU(inplace=True),
        }
        return activations.get(activation.lower(), nn.ReLU(inplace=True))

    def _make_conv_block(
        self,
        in_channels: int,
        out_channels: int,
        num_conv_layers: int,
        kernel_size: int,
        pool_size: int,
        dropout_rate: float,
        current_size: int,
    ) -> tuple:
        """
        构建单个卷积块
        """

        layers = []

        layers.extend(
            [
                nn.Conv2d(
                    in_channels, out_channels, kernel_size, padding=kernel_size // 2
                ),
                nn.BatchNorm2d(out_channels) if self.use_batch_norm else nn.Identity(),
                self._get_activation_fn(self.activation.__class__.__name__.lower()),
            ]
        )

        for _ in range(num_conv_layers - 1):
            layers.extend(
                [
                    nn.Conv2d(
                        out_channels,
                        out_channels,
                        kernel_size,
                        padding=kernel_size // 2,
                    ),
                    nn.BatchNorm2d(out_channels)
                    if self.use_batch_norm
                    else nn.Identity(),
                    self._get_activation_fn(self.activation.__class__.__name__.lower()),
                ]
            )

        if pool_size > 1:
            layers.append(nn.MaxPool2d(pool_size, pool_size))
            current_size = current_size // pool_size

        if dropout_rate > 0:
            layers.append(nn.Dropout2d(dropout_rate))

        return nn.Sequential(*layers), out_channels, current_size

    def _calculate_feature_size(self, channels: int, size: int) -> int:
        """
        计算全连接层输入尺寸
        """

        if self.global_pool in ["adaptive_avg", "adaptive_max"]:
            return channels
        else:
            return channels * size * size

    def _make_fc_layers(
        self,
        input_size: int,
        hidden_layers: List[int],
        output_size: int,
        dropout_rates: List[float],
    ) -> nn.Module:
        """
        构建全连接层
        """

        layers = []
        current_size = input_size

        for i, hidden_size in enumerate(hidden_layers):
            layers.extend(
                [
                    nn.Linear(current_size, hidden_size),
                    self._get_activation_fn(self.activation.__class__.__name__.lower()),
                    nn.Dropout(dropout_rates[i] if i < len(dropout_rates) else 0.5),
                ]
            )
            current_size = hidden_size

        layers.append(nn.Linear(current_size, output_size))

        return nn.Sequential(*layers)

    def _initialize_weights(self):
        """
        初始化网络权重
        """

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播
        """

        for conv_block in self.conv_blocks:
            x = conv_block(x)

        if self.global_pool == "adaptive_avg":
            x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
        elif self.global_pool == "adaptive_max":
            x = nn.functional.adaptive_max_pool2d(x, (1, 1))
        elif self.global_pool == "avg":
            x = nn.functional.avg_pool2d(x, x.size()[2:])
        elif self.global_pool == "max":
            x = nn.function.max_pool2d(x, x.size()[2:])

        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

    def get_model_info(self) -> Dict[str, Any]:
        """
        获取模型信息
        """

        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)

        return {
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "conv_blocks": len(self.conv_blocks),
            "feature_size": self.feature_size,
            "activation": self.activation.__class__.__name__,
            "use_batch_norm": self.use_batch_norm,
            "global_pool": self.global_pool,
        }

# 5、模型训练与测试类定义


In [ ]:
class Trainer:
    """
    训练器类
    """

    def __init__(self, model, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.model = model.to(device)
        self.device = device
        self.train_losses = []
        self.train_accs = []
        self.val_losses = []
        self.val_accs = []

    def train_epoch(self, train_loader, criterion, optimizer, verbose=False):
        """
        训练一个epoch
        """

        self.model.train()
        total_loss = 0
        correct = 0
        total = 0

        progress_bar = tqdm.tqdm(train_loader, desc="Training")
        for data, target in progress_bar:
            data, target = data.to(self.device), target.to(self.device)

            optimizer.zero_grad()
            output = self.model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

            if verbose:
                progress_bar.set_description(
                    f"Training - Loss: {loss.item():.3f}, "
                    f"Acc: {100.0 * correct / total:.2f}%"
                )

        avg_loss = total_loss / len(train_loader)
        accuracy = 100.0 * correct / total

        return avg_loss, accuracy

    def evaluate(self, val_loader, criterion):
        """
        评估模型
        """

        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                loss = criterion(output, target)

                total_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        avg_loss = total_loss / len(val_loader)
        accuracy = 100.0 * correct / total

        return avg_loss, accuracy

    def train(
        self,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        scheduler=None,
        max_epochs=10,
    ):
        """
        完整训练过程
        """

        print(f"Training on device: {self.device}")
        best_val_loss = float("inf")
        patience_counter = 0

        for epoch in range(max_epochs):
            train_loss, train_acc = self.train_epoch(train_loader, criterion, optimizer)
            val_loss, val_acc = self.evaluate(val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= 20:  # Early stopping
                logger.info(f"Early stopping at epoch {epoch + 1}")
                break

            self.train_losses.append(train_loss)
            self.train_accs.append(train_acc)
            self.val_losses.append(val_loss)
            self.val_accs.append(val_acc)

            print(
                f"Epoch [{epoch + 1}/{max_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
            )

            if scheduler:
                scheduler.step()

        return {
            "train_losses": self.train_losses,
            "train_accs": self.train_accs,
            "val_losses": self.val_losses,
            "val_accs": self.val_accs,
        }

    def test(self, test_loader, criterion):
        """
        测试模型
        """

        test_loss, test_acc = self.evaluate(test_loader, criterion)
        print(
            f"\nTest Results - Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}%"
        )

        return test_loss, test_acc

# 6、可视化函数定义


In [ ]:
def plot_training_curves(train_losses, val_losses, train_accs, val_accs):
    """
    绘制训练验证曲线
    """

    epochs = range(1, len(train_losses) + 1)

    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), dpi=300)

    # 损失曲线
    ax1.plot(epochs, train_losses, "b-", label="Training Loss", marker=".")
    ax1.plot(epochs, val_losses, "r-", label="Validation Loss", marker=".")
    ax1.set_xticks(np.linspace(1, len(train_losses) + 1, 10, dtype=int))
    ax1.set_title("Training and Validation Loss", fontsize=14, color="purple")
    ax1.set_xlabel("Epochs", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.margins(x=0.02, y=0.1)
    ax1.legend()
    ax1.grid(True, alpha=0.1)

    # 准确率曲线
    ax2.plot(epochs, train_accs, "b-", label="Training Accuracy", marker=".")
    ax2.plot(epochs, val_accs, "r-", label="Validation Accuracy", marker=".")
    ax2.set_xticks(np.linspace(1, len(train_losses) + 1, 10, dtype=int))
    ax2.set_title("Training and Validation Accuracy", fontsize=14, color="purple")
    ax2.set_xlabel("Epochs", fontsize=12)
    ax2.set_ylabel("Accuracy (%)", fontsize=12)
    ax2.margins(x=0.02, y=0.1)
    ax2.legend()
    ax2.grid(True, alpha=0.1)

    plt.tight_layout()
    plt.show()

# 7、实验与分析


## 7.1、数据加载


In [ ]:
loader = CIFAR10Loader()

# 加载数据
loader.extract_and_load_data()

# 显示统计信息
loader.show_statistics()

# 可视化数据集
loader.visualize_dataset()

# 获取PyTorch数据加载器
train_loader, val_loader, test_loader = loader.get_data_loaders()

logger.info(f"Train loader: {len(train_loader)} batches")
logger.info(f"Val loader: {len(val_loader)} batches")
logger.info(f"Test loader: {len(test_loader)} batches")

## 7.2、多层感知机模型实验

### 7.2.1、默认配置

In [ ]:
# 超参数配置
config = {
    "batch_size": 64,
    "learning_rate": 1e-3,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

print("=" * 25)
print("Training MLP Model")
print("=" * 25)

mlp_model = MLP()
mlp_trainer = Trainer(mlp_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

mlp_results = mlp_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试MLP
_, mlp_test_acc = mlp_trainer.test(test_loader, criterion)

# 绘制MLP训练曲线
print("\nMLP Training Curves:")
plot_training_curves(
    mlp_results["train_losses"],
    mlp_results["val_losses"],
    mlp_results["train_accs"],
    mlp_results["val_accs"],
)
print(f"MLP Test Accuracy: {mlp_test_acc:.2f}%.")

&emsp;&emsp;**默认的MLP模型采用了全连接神经网络结构，适配CIFAR-10数据集，其输入维度为3072，对应于32x32像素RGB图像展平后的向量。模型包含三个隐藏层，神经元数量分别为512、256和128，使用ReLU激活函数引入非线性变换，输出层为10个神经元对应10个类别。模型整合了多项正则化技术以防止过拟合，包括BatchNorm层来加速训练并提高稳定性，以及Dropout层，对应的丢弃率分别为0.1、0.2、0.3和0.4，随机禁用神经元以增强泛化能力。权重初始化策略针对ReLU激活函数进行了优化，隐藏层使用Kaiming He初始化，输出层采用较小标准差（0.01）的正态分布初始化以确保训练初期稳定性。**

&emsp;&emsp;**在CIFAR-10数据集上的训练结果显示，模型在训练集上的性能持续提升，损失从1.7163降至0.7150，准确率从38.69%升至74.52%，但验证集和测试集性能改善有限，验证准确率在46%至58%之间波动，最终测试准确率为57.89%，这表明模型出现了明显的过拟合。训练损失持续下降而验证损失早期下降后趋于波动甚至回升，例如epoch 10后验证损失在1.25-1.38间震荡，验证准确率始终显著低于训练准确率，且差距随训练扩大，进一步证实了过拟合现象。这种性能瓶颈源于MLP架构的本质局限性：其全连接结构需要将图像展平为向量，破坏了像素间的空间局部性和二维结构信息，比如边缘、纹理等局部特征，导致模型难以有效学习图像的空间层次特征。尽管使用了BatchNorm和Dropout等正则化技术，MLP在处理像CIFAR-10这样的图像数据时仍因参数量大且缺乏空间感知能力而容易过拟合**

### 7.2.2 更换激活函数

In [ ]:
class MLP(nn.Module):
    """
    多层感知机模型
    """

    def __init__(
        self,
        input_size=3072,
        hidden_sizes=[512, 256, 128],
        num_classes=10,
        dropouts=[0.1, 0.2, 0.3, 0.4],
        use_batch_norm=True,
    ):
        super(MLP, self).__init__()

        self.use_batch_norm = use_batch_norm
        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropouts])
        self.batch_norms = nn.ModuleList() if use_batch_norm else None

        self.layers.append(nn.Linear(input_size, hidden_sizes[0]))
        if use_batch_norm:
            self.batch_norms.append(nn.BatchNorm1d(hidden_sizes[0]))

        for i in range(len(hidden_sizes) - 1):
            self.layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))
            if use_batch_norm:
                self.batch_norms.append(nn.BatchNorm1d(hidden_sizes[i + 1]))

        self.layers.append(nn.Linear(hidden_sizes[-1], num_classes))
        self.activation = nn.GELU()

        self._initialize_weights()

    def _initialize_weights(self):
        """
        权重初始化
        """

        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    if m is self.layers[-1]:
                        nn.init.normal_(m.bias, mean=0.0, std=0.02)
                    else:
                        nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.BatchNorm1d):
                if m.weight is not None:
                    nn.init.constant_(m.weight, 1.0)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            if self.use_batch_norm and i < len(self.batch_norms):
                x = self.batch_norms[i](x)
            x = self.activation(x)
            x = self.dropouts[i](x)

        x = self.layers[-1](x)
        return x


# 超参数配置
config = {
    "batch_size": 64,
    "learning_rate": 1e-3,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

print("=" * 25)
print("Training MLP Model")
print("=" * 25)

mlp_model = MLP()
mlp_trainer = Trainer(mlp_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

mlp_results = mlp_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试MLP
_, mlp_test_acc = mlp_trainer.test(test_loader, criterion)

# 绘制MLP训练曲线
print("\nMLP Training Curves:")
plot_training_curves(
    mlp_results["train_losses"],
    mlp_results["val_losses"],
    mlp_results["train_accs"],
    mlp_results["val_accs"],
)
print(f"MLP Test Accuracy: {mlp_test_acc:.2f}%.")

&emsp;&emsp;**该MLP模型采用多层感知机架构，适配CIFAR-10图像分类任务，输入维度为3072对应32x32像素RGB图像展平后的向量，包含三个隐藏层，神经元数量分别为512、256和128。使用GELU激活函数，输出层为10个神经元对应10个类别，模型整合了BatchNorm层加速训练并提高稳定性，以及递增Dropout率0.1、0.2、0.3、0.4，防止过拟合。权重初始化采用正态分布标准差0.02，适配GELU激活函数特性。**

&emsp;&emsp;**在CIFAR-10数据集上训练32个epoch后因早停机制终止，最终测试准确率为57.37%，训练过程显示明显过拟合现象训练损失从1.6785持续降至0.6484，训练准确率从40.27%提升至76.74%，而验证损失在1.25至1.4之间波动。验证准确率停滞在58%左右，训练与验证性能差距随epoch增加而扩大，表明模型过度记忆训练数据特征，缺乏泛化能力，这种性能瓶颈源于MLP架构本质缺陷展平操作破坏图像空间结构全，连接层难以有效学习局部特征。**

&emsp;&emsp;**相比先前使用ReLU激活函数的MLP模型，本模型主要改进在于采用GELU激活函数和适配的权重初始化，然而性能不仅没有提升反而下降，测试准确率仅从57.89%降至57.37%，训练过程同样呈现过拟合，验证准确率始终低于60%，两者共同表明MLP架构在处理图像数据的根本局限，无论激活函数如何更换，全连接网络难以捕获空间层次特征无法突破60%准确率瓶颈。**

### 7.2.3、增加层数

In [ ]:
# 超参数配置
config = {
    "batch_size": 64,
    "learning_rate": 1e-3,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

print("=" * 25)
print("Training MLP Model")
print("=" * 25)

mlp_model = MLP(
    hidden_sizes=[2048, 1024, 512, 256, 128], dropouts=[0.1, 0.1, 0.2, 0.3, 0.4, 0.5]
)
mlp_trainer = Trainer(mlp_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

mlp_results = mlp_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试MLP
_, mlp_test_acc = mlp_trainer.test(test_loader, criterion)

# 绘制MLP训练曲线
print("\nMLP Training Curves:")
plot_training_curves(
    mlp_results["train_losses"],
    mlp_results["val_losses"],
    mlp_results["train_accs"],
    mlp_results["val_accs"],
)
print(f"MLP Test Accuracy: {mlp_test_acc:.2f}%.")

&emsp;&emsp;**这个MLP模型适配CIFAR-10图像分类任务，输入维度为3072对应32x32像素RGB图像展平后的向量，包含五个隐藏层，神经元数量依次为2048、1024、512、256和128，输出层为10个神经元对应10个类别。模型使用GELU作为激活函数，整合了BatchNorm1d层以加速训练并提高稳定性，同时配置了六个Dropout层，dropout率依次为0.1、0.1、0.2、0.3、0.4和0.5，用于抑制过拟合。权重初始化采用正态分布，均值为0.0、标准差为0.02，偏置初始化根据层的位置有所区别，除输出层外其余隐藏层偏置初始化为常数0.0。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第41个epoch时因早停机制终止，训练过程中训练损失从初始的1.7472持续下降至0.5741，训练准确率从37.35%逐步提升至79.83%，显示模型对训练数据的拟合能力不断增强。验证损失则在1.2454至1.4965之间波动，未出现明显的持续下降趋势，验证准确率最高达到58.08%且整体维持在55%至58%之间，最终测试损失为1.4531，测试准确率为57.50%。训练损失与验证损失的差距随epoch增加而逐渐扩大，训练准确率与验证准确率的差距也持续存在，表明模型存在明显的过拟合现象，对未见过的测试数据泛化能力较弱。**

&emsp;&emsp;**与之前使用三层隐藏层MLP模型相比，这个模型增加了隐藏层数量至五层且初始隐藏层神经元数量大幅提升，同时增加了Dropout层的数量并提高了后期Dropout率。从训练表现来看，这个模型的训练损失下降到了更低水平，训练准确率也提升了约3个百分点，说明更复杂的网络结构增强了对训练数据的拟合能力。但在泛化能力方面，两者的验证准确率最高值相近，这个模型的验证损失后期波动更大且整体偏高，测试准确率仅略高0.13个百分点，并未出现明显的泛化能力提升。这一结果表明增加MLP的层数和神经元数量虽然能提高模型对训练数据的拟合程度，但由于MLP架构本身通过展平操作破坏了图像的空间结构，全连接层难以有效学习局部特征，即便增加Dropout层的数量和强度，也无法突破架构本身的局限，难以显著改善泛化能力。**

### 7.2.4、更换优化器

#### 7.2.4.1、SGD优化器

In [ ]:
# 超参数配置
config = {
    "batch_size": 64,
    "learning_rate": 1e-3,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

print("=" * 25)
print("Training MLP Model")
print("=" * 25)

mlp_model = MLP(
    hidden_sizes=[2048, 1024, 512, 256, 128], dropouts=[0.1, 0.1, 0.2, 0.3, 0.4, 0.5]
)
mlp_trainer = Trainer(mlp_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    mlp_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

mlp_results = mlp_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试MLP
_, mlp_test_acc = mlp_trainer.test(test_loader, criterion)

# 绘制MLP训练曲线
print("\nMLP Training Curves:")
plot_training_curves(
    mlp_results["train_losses"],
    mlp_results["val_losses"],
    mlp_results["train_accs"],
    mlp_results["val_accs"],
)
print(f"MLP Test Accuracy: {mlp_test_acc:.2f}%.")

&emsp;&emsp;**这个MLP模型适配CIFAR-10图像分类任务，输入维度为3072对应32x32像素RGB图像展平后的向量，包含五个隐藏层，神经元数量依次为2048、1024、512、256和128，输出层为10个神经元对应10个类别。模型采用GELU作为激活函数，整合了BatchNorm1d层以提升训练稳定性和收敛速度，同时配置了六个Dropout层，dropout率依次为0.1、0.1、0.2、0.3、0.4和0.5以抑制过拟合。优化器选用SGD，学习率设置为1e-3，权重衰减为1e-4以实现正则化，学习率调度器采用StepLR，每20个epoch将学习率乘以0.5进行衰减。**

&emsp;&emsp;**该模型在CIFAR-10数据集上完成了100个epoch的训练，训练过程中训练损失从初始的2.2652持续下降至最终的1.1622，下降趋势前期较为明显，后期逐渐平缓；训练准确率从15.43%逐步提升至58.81%，整体呈现稳步上升态势但后期增速放缓。验证损失初始为2.1977，之后在1.31至1.50之间波动，最终稳定在1.3182；验证准确率从25.56%提升至54.48%，期间无明显持续上升或下降趋势，始终在53%至55%之间小幅波动。最终测试损失为1.3016，测试准确率为53.72%，表明模型对训练数据有一定拟合能力，但泛化到测试数据的表现较为一般。**

&emsp;&emsp;**与之前使用Adam优化器的同结构MLP模型相比，此次模型的核心变化是将优化器更换为SGD，这带来了多方面差异。从收敛速度来看，之前使用Adam的模型在41个epoch时因早停机制终止训练，而此次使用SGD的模型需跑满100个epoch才能达到相对稳定的状态，收敛速度明显更慢。从最终性能来看，之前模型的测试准确率为57.50%，此次模型测试准确率降至53.72%，性能有所下降；训练损失方面，之前模型最终训练损失低至0.5741，此次模型最终训练损失为1.1622，拟合程度也更低。从过拟合情况来看，之前模型的训练损失与验证损失差距较大，过拟合现象更明显，而此次模型因SGD收敛速度慢且拟合程度较低，训练与验证指标的差距相对更小，过拟合程度有所缓解。**

#### 7.2.3.2、AdamW优化器

In [ ]:
# 超参数配置
config = {
    "batch_size": 64,
    "learning_rate": 1e-3,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

print("=" * 25)
print("Training MLP Model")
print("=" * 25)

mlp_model = MLP(
    hidden_sizes=[2048, 1024, 512, 256, 128], dropouts=[0.1, 0.1, 0.2, 0.3, 0.4, 0.5]
)
mlp_trainer = Trainer(mlp_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    mlp_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

mlp_results = mlp_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试MLP
_, mlp_test_acc = mlp_trainer.test(test_loader, criterion)

# 绘制MLP训练曲线
print("\nMLP Training Curves:")
plot_training_curves(
    mlp_results["train_losses"],
    mlp_results["val_losses"],
    mlp_results["train_accs"],
    mlp_results["val_accs"],
)
print(f"MLP Test Accuracy: {mlp_test_acc:.2f}%.")

&emsp;&emsp;**该MLP模型用于CIFAR-10图像分类任务，输入维度为3072，对应32x32像素RGB图像展平后的向量，网络结构包含五个隐藏层，神经元数量依次为2048、1024、512、256和128，输出层设置10个神经元以匹配数据集的10个类别。模型采用GELU作为激活函数，同时整合了BatchNorm1d层，用于提升训练过程的稳定性并加速收敛，还配置了六个Dropout层，dropout率依次为0.1、0.1、0.2、0.3、0.4和0.5，通过随机失活部分神经元来抑制过拟合。优化器选用AdamW，学习率设置为1e-3，权重衰减参数为1e-4以增强正则化效果，学习率调度器采用StepLR，每20个epoch将学习率乘以0.5进行衰减。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第29个epoch时因早停机制终止，训练过程中训练损失呈现快速下降趋势，从初始的1.7260持续降至0.1722，训练准确率则从38.21%稳步提升至94.21%，表明模型对训练数据的拟合能力极强且收敛速度快。与之形成鲜明对比的是，验证损失从初始的1.5198逐渐上升至2.1057，验证准确率始终在45.44%至58.54%之间波动，未随训练进程出现明显提升，最高仅达到58.54%。最终测试结果显示，测试损失为2.1480，测试准确率为57.45%，训练损失与验证损失、训练准确率与验证准确率的差距随epoch增加不断扩大，凸显出模型存在严重的过拟合现象，对未见过的测试数据泛化能力较弱。**

&emsp;&emsp;**与之前讨论的同结构MLP模型相比，核心差异在于优化器的选择，此前分别使用过SGD和Adam优化器。从训练进程来看，使用AdamW的该模型收敛速度最快，仅需29个epoch就触发早停，而使用SGD的模型需跑满100个epoch，使用Adam的模型则在41个epoch触发早停。训练指标方面，该模型的训练损失下降幅度最大，最终训练损失远低于另外两个模型，训练准确率也最高，达到94.21%，而使用SGD的模型最终训练准确率仅为58.81%，使用Adam的模型约为79.83%。但从泛化能力来看，该模型的验证损失上升最为明显，过拟合程度比另外两个模型更严重，测试准确率则与使用Adam的模型（57.50%）接近，略高于使用SGD的模型（53.72%），整体未实现显著突破。**

&emsp;&emsp;**产生这种变化的核心原因在于AdamW优化器的特性，AdamW结合了Adam自适应学习率的优势和更有效的权重衰减机制，能为不同参数分配更合理的学习率，加速参数更新，因此模型收敛速度快、训练损失下降迅速、训练准确率提升明显。但自适应学习率优化器本身容易导致模型过度拟合训练数据，加上该MLP模型包含五个隐藏层且神经元数量多，网络容量大，即使配置了多个Dropout层，也难以完全抑制过拟合，使得验证损失持续上升、验证准确率停滞不前。与SGD优化器相比，SGD采用固定学习率，参数更新更为保守，虽然收敛慢、训练拟合程度低，但过拟合程度相对较轻；与Adam优化器相比，AdamW的权重衰减机制更直接有效，一定程度上缓解了Adam可能出现的泛化能力不足问题，但未能突破MLP架构的固有局限，即图像展平操作破坏了空间结构，全连接层无法有效提取局部特征，导致所有MLP模型的测试准确率都难以突破60%，始终在53%至58%之间徘徊。**

## 7.3、卷积神经网络模型实验

### 7.3.1、较少层数

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 32,
            "num_conv_layers": 1,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 64,
            "num_conv_layers": 1,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [128],
    "dropout_rates": [0.1, 0.1, 0.3],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试ConvNet
_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

# 绘制ConvNet训练曲线
print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型用于CIFAR-10图像分类任务，输入通道数为3，对应RGB图像，输入尺寸为32×32，最终输出10个类别对应数据集类别数。模型包含两个卷积块，第一个卷积块配置32个输出通道，内含1个3×3卷积层、BatchNorm2d层、ReLU激活函数、2×2最大池化层和0.1概率的Dropout2d层；第二个卷积块配置64个输出通道，结构与第一个卷积块一致，同样包含3×3卷积层、BatchNorm2d层、ReLU激活函数、2×2最大池化层和0.1概率的Dropout2d层。卷积块之后采用自适应平均池化，将特征图压缩为1×1大小以保留通道维度，随后连接一个全连接层模块，包含128个神经元的线性层、ReLU激活函数、0.3概率的Dropout层，最后通过线性层输出10个类别结果。模型可训练参数均29194，整体结构通过卷积提取空间特征、池化降低维度、正则化抑制过拟合，形成适配图像任务的特征学习流程。**

&emsp;&emsp;**该模型在CIFAR-10数据集上完成100个epoch的训练，训练过程中训练损失呈现稳步下降趋势，从初始的1.9502逐渐降至1.2562，下降幅度均匀且后期趋于平缓，无明显停滞或反弹；训练准确率从24.57%逐步提升至54.52%，整体增速稳定，未出现快速飙升后停滞的情况。验证损失从1.7546持续下降至1.1046，验证准确率从30.96%提升至60.64%，训练损失与验证损失的差距始终保持在较小范围，训练准确率与验证准确率的差距也未随epoch增加而显著扩大。最终测试结果显示，测试损失为1.1150，测试准确率达到60.88%，整体表现出良好的收敛性和泛化能力，未出现明显的过拟合现象。**

&emsp;&emsp;**与之前讨论的各类MLP模型相比，该ConvNet模型在性能和结构上均有显著变化。结构方面，MLP模型依赖全连接层处理展平后的图像向量，丢失了图像的空间结构信息，且隐藏层神经元数量多，如含2048、1024等神经元的多层结构，导致参数规模远大于ConvNet的29194个参数；而ConvNet通过卷积层提取局部空间特征，配合池化层降低维度，在保留关键信息的同时减少参数和计算量。训练表现方面，此前MLP模型的测试准确率最高仅为57.5%左右，且普遍存在严重过拟合，如使用AdamW的MLP训练准确率高达94.21%但验证准确率仅58.54%；ConvNet的测试准确率突破60%达到60.88%，且训练与验证指标差距小，过拟合程度远低于MLP模型，表现出更优的泛化能力。**

&emsp;&emsp;**产生这种变化的核心原因在于ConvNet的结构设计更适配图像数据特性，卷积层能够捕捉图像的局部空间关联特征，如边缘、纹理等，这些特征是图像分类的关键信息，而MLP将图像展平为一维向量后，完全丢失了像素间的空间位置关系，只能学习全局像素的统计关联，难以有效提取图像特有的结构特征。同时，ConvNet中的池化层通过下采样减少特征图尺寸，不仅降低了后续层的参数数量和计算复杂度，还能增强模型对图像轻微位移的鲁棒性，进一步提升泛化能力。此外，ConvNet的参数规模更小，仅29194个可训练参数，远低于MLP模型的庞大参数量，减少了模型过拟合的风险，再配合BatchNorm层稳定训练过程、Dropout层抑制过拟合，使得模型能够在收敛到合理训练效果的同时，保持良好的泛化性能，最终实现测试准确率的突破和过拟合现象的缓解。**

### 7.3.2、增加模型卷积块数量

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 32,
            "num_conv_layers": 1,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 64,
            "num_conv_layers": 1,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 128,
            "num_conv_layers": 1,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [128],
    "dropout_rates": [0.1, 0.1, 0.3],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试ConvNet
_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

# 绘制ConvNet训练曲线
print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型适配CIFAR-10图像分类任务，输入通道数为3，对应RGB图像，输入尺寸为32×32，输出10个类别以匹配数据集类别数。模型包含三个卷积块，第一个卷积块配置32个输出通道，内含1个3×3卷积层、BatchNorm2d层、ReLU激活函数、2×2最大池化层和0.1概率的Dropout2d层；第二个卷积块配置64个输出通道，结构与第一个卷积块一致，同样包含3×3卷积层、BatchNorm2d层、ReLU激活函数、2×2最大池化层和0.1概率的Dropout2d层；第三个卷积块配置128个输出通道，结构与前两个卷积块相同，仅输出通道数不同。卷积块之后采用自适应平均池化，将特征图压缩为1×1大小以保留128个通道的特征信息，随后连接全连接层模块，该模块包含128个神经元的线性层、ReLU激活函数、0.3概率的Dropout层，最后通过线性层输出10个类别结果。模型总参数与可训练参数均为111498，整体通过多卷积块逐步提取图像特征，配合正则化模块保障训练过程的稳定性。**

&emsp;&emsp;**该模型在CIFAR-10数据集上完成100个epoch的训练，训练损失呈现持续且平稳的下降趋势，从初始的1.8906逐步降至最终的0.9241，下降过程中无明显停滞或反弹现象，显示模型的训练过程稳定且有效。训练准确率从26.04%稳步提升至67.34%，增速在前期稍快，后期逐渐平缓，说明模型对训练数据的拟合能力逐步增强且未出现过早饱和的情况。验证损失从1.6458持续下降至0.7826，验证准确率从36.04%提升至72.60%，训练损失与验证损失的差距始终保持在较小范围，训练准确率与验证准确率的差距也未随epoch增加而显著扩大，未出现明显的过拟合现象。最终的测试结果显示，测试损失为0.7889，测试准确率达到71.98%，测试性能与验证性能较为接近，表明模型具有良好的泛化能力。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构和训练表现上均有显著变化。结构方面，此前的MLP模型依赖全连接层处理展平后的图像向量，完全丢失了图像的空间结构信息，且参数规模庞大；而该ConvNet模型通过三个卷积块提取图像的空间特征，参数数量111498虽多于仅有两个卷积块的ConvNet模型（29194个参数），但远少于复杂MLP模型的参数量，同时该模型比两卷积块ConvNet多一个128通道的卷积块，特征提取能力更强。训练表现方面，MLP模型的测试准确率最高仅为57.5%左右，且普遍存在严重的过拟合问题；两卷积块ConvNet模型的测试准确率为60.88%；而该模型的测试准确率突破70%达到71.98%，性能提升明显。此外，该模型的训练收敛速度快于MLP模型和两卷积块ConvNet模型，前期训练损失和准确率的提升更为迅速，且验证性能与训练性能的一致性更好，过拟合程度远低于MLP模型，也轻于两卷积块ConvNet模型。**

&emsp;&emsp;**产生这种变化的核心原因在于该模型增加了卷积块的数量并提升了特征通道数，三个卷积块能够逐步提取图像从浅层到深层的特征，第一个卷积块可捕捉边缘、纹理等基础特征，第二个卷积块能组合基础特征形成局部结构特征，第三个128通道的卷积块则可进一步提取更复杂的全局关联特征，相比仅有两个卷积块的模型，能保留更丰富的图像信息，更符合图像分类任务对多层特征的需求。同时，虽然模型的参数数量有所增加，但通过BatchNorm2d层稳定了各层输入的分布，加速了训练收敛并缓解了梯度消失问题，配合Dropout2d层随机失活部分特征图，有效抑制了参数增加可能带来的过拟合风险。此外，卷积层固有的空间特征提取能力本身就优于MLP模型的全连接层，多卷积块的设计进一步强化了这一优势，使得模型能够更高效地学习图像特有的空间结构信息，最终实现了测试准确率的显著提升和泛化能力的优化。**

### 7.3.3、增加每个卷积块中连续堆叠的卷积层的数量

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 32,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 64,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 128,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [128],
    "dropout_rates": [0.1, 0.1, 0.3],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

# 测试ConvNet
_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

# 绘制ConvNet训练曲线
print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型用于CIFAR-10图像分类任务，输入通道数为3，对应RGB图像，输入尺寸为32×32，输出10个类别以匹配数据集类别数。模型包含三个卷积块，每个卷积块内部均配置3个3×3卷积层，第一个卷积块输出通道为32，第二个为64，第三个为128，每个卷积层后均跟随BatchNorm2d层和ReLU激活函数，每个卷积块末尾设置2×2最大池化层以降低特征图尺寸，同时第一个和第二个卷积块后配置0.1概率的Dropout2d层。卷积块之后采用自适应平均池化，将最终特征图压缩为1×1大小，保留128个通道的特征信息，随后连接全连接层模块，该模块包含1个128神经元的线性层、ReLU激活函数和0.3概率的Dropout层，最后通过线性层输出10个类别结果。模型总参数与可训练参数均为499914，整体结构通过多卷积层堆叠的卷积块实现深层特征提取，配合正则化模块保障训练稳定性。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第44个epoch时因早停机制终止，训练过程中训练损失呈现快速且显著的下降趋势，从初始的1.7654迅速降至第40个epoch的0.2141，训练准确率从29.59%稳步提升至92.60%，前期每epoch损失下降幅度较大，后期下降速度逐渐放缓，显示模型对训练数据的拟合能力极强且收敛效率高。验证损失从1.5366下降至第40个epoch的0.5851，验证准确率从38.96%提升至84.22%，训练损失与验证损失的差距虽随训练进程略有扩大，但始终保持在合理范围，未出现严重的过拟合现象。最终测试结果显示，测试损失为0.6092，测试准确率达到84.12%，测试性能与验证性能高度接近，表明模型具有优异的泛化能力。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构和训练表现上均实现了显著突破。结构方面，此前的MLP模型依赖全连接层处理展平向量，丢失空间信息且参数冗余；两卷积块ConvNet模型（每个块1个卷积层）参数仅29194，特征提取能力有限；三卷积块但每个块1个卷积层的ConvNet模型参数为111498，虽优于两卷积块模型，但深层特征捕捉仍不足。而该模型每个卷积块包含3个卷积层，参数规模增至499914，通过多层卷积堆叠实现更精细的特征提取。训练表现方面，MLP模型测试准确率最高仅57.5%，两卷积块ConvNet为60.88%，三卷积块单卷积层ConvNet为71.98%，该模型则突破84%达到84.12%，性能提升幅度显著。同时，该模型收敛速度远快于前几类模型，前10个epoch训练准确率即可从29.59%提升至74.13%，且过拟合控制效果优于MLP模型，泛化能力更优。**

&emsp;&emsp;**产生这种变化的核心原因在于每个卷积块内部的多卷积层堆叠设计，每个卷积块中的3个卷积层能够在不急于下采样的情况下，逐步细化当前通道的特征信息，例如第一个卷积块的3个32通道卷积层可反复提取图像的边缘、纹理等基础特征，捕捉更丰富的局部细节，避免单卷积层提取特征过于粗糙的问题。多卷积层堆叠还能实现特征的层级组合，浅层卷积层提取基础特征，深层卷积层将基础特征组合为更复杂的结构特征，更符合图像分类任务对多层级特征的需求。此外，尽管模型参数数量大幅增加，但BatchNorm2d层有效稳定了各层输入数据分布，缓解了深层网络的梯度消失问题，Dropout层则通过随机失活部分特征图抑制过拟合风险，再配合Adam优化器的自适应学习率和StepLR学习率调度，进一步提升了训练效率和模型稳定性，最终实现了测试准确率的大幅突破和泛化能力的优化。**

### 7.3.4、改用AdamW优化器

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 32,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 64,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 128,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [128],
    "dropout_rates": [0.1, 0.1, 0.3],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型适配CIFAR-10图像分类任务，输入通道数为3，对应RGB图像，输入尺寸为32×32，输出10个类别以匹配数据集类别数。模型包含三个卷积块，每个卷积块内部均堆叠3个3×3卷积层，第一个卷积块输出通道为32，第二个为64，第三个为128，每个卷积层后均紧跟BatchNorm2d层和ReLU激活函数，以稳定训练过程并增强非线性表达能力。每个卷积块末尾设置2×2最大池化层，用于降低特征图尺寸、减少计算量并提升特征鲁棒性，其中前两个卷积块后还配置0.1概率的Dropout2d层以抑制过拟合。卷积块之后采用自适应平均池化操作，将最终特征图压缩为1×1大小，保留128个通道的特征信息，随后连接全连接层模块，该模块包含1个128神经元的线性层、ReLU激活函数和0.3概率的Dropout层，最后通过线性层输出10个类别结果。模型总参数与可训练参数均为499914，整体通过多层卷积堆叠实现深层特征提取，配合正则化模块保障训练稳定性与泛化能力。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第42个epoch时因早停机制终止，训练过程呈现出高效的收敛趋势。训练损失从初始的1.7674持续下降至最终的0.2359，下降幅度前期较为显著，后期逐渐平缓，表明模型对训练数据的拟合能力逐步增强且未出现过早饱和；训练准确率从29.11%稳步提升至92.08%，整体增速均匀，反映出模型学习过程稳定。验证损失从1.5173下降至0.5862，验证准确率从39.60%提升至83.70%，训练损失与验证损失的差距始终保持在合理范围，未出现明显扩大趋势，训练准确率与验证准确率的差异也相对稳定，说明模型未产生严重的过拟合现象。最终测试结果显示，测试损失为0.6294，测试准确率达到83.01%，测试性能与验证性能高度接近，进一步证明模型具有良好的泛化能力。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上与此前采用三个卷积块（每块3个卷积层）的ConvNet完全一致，参数规模均为499914，核心差异仅在于优化器从Adam更换为AdamW；而与结构更简单的模型相比，如MLP（最高测试准确率57.5%）、两卷积块ConvNet（60.88%）、三卷积块单卷积层ConvNet（71.98%），该模型的测试准确率仍显著更高，保持了深层卷积结构的性能优势。与同结构采用Adam优化器的ConvNet相比，该模型的测试准确率从84.12%略降至83.01%，训练准确率从92.60%小幅下降至92.08%，早停 epoch 从44提前至42，整体性能呈现轻微波动，但未出现大幅下滑，仍处于较高水平。**

&emsp;&emsp;**产生这种细微变化的核心原因在于AdamW与Adam优化器的权重衰减机制差异。Adam优化器将权重衰减融入梯度更新过程，等效于对参数施加L2正则化，这种方式可能导致权重衰减效果随梯度大小波动；而AdamW优化器则将梯度更新与权重衰减分离，直接对模型权重进行独立衰减，权重衰减的力度和稳定性更强。在该模型中，AdamW更严格的权重衰减机制轻微抑制了模型对训练数据的拟合程度，使得训练准确率和早停 epoch 略低于Adam版本；同时，由于模型本身已通过多卷积层堆叠和Dropout等正则化模块有效控制过拟合，AdamW额外的权重衰减未能进一步提升泛化能力，反而因拟合程度轻微降低导致测试准确率小幅下降。不过，这种差异并未掩盖深层卷积结构的核心优势，模型仍远优于结构更简单的模型，说明多卷积层堆叠对图像特征的高效提取才是性能提升的关键，优化器的影响更多体现在性能的细微调优上。**

### 7.3.5、更换激活函数

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 32,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 64,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 128,
            "num_conv_layers": 3,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "activation": "gelu",
    "fc_layers_config": [128],
    "dropout_rates": [0.1, 0.1, 0.3],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型用于CIFAR-10图像分类任务，输入通道数为3，对应RGB图像，输入尺寸为32×32，输出10个类别以匹配数据集类别数。模型包含三个卷积块，每个卷积块内部均堆叠3个3×3卷积层，第一个卷积块输出通道为32，第二个为64，第三个为128，每个卷积层后均跟随BatchNorm2d层和GELU激活函数，以稳定训练分布并增强非线性表达的平滑性。每个卷积块末尾设置2×2最大池化层，用于降低特征图尺寸、减少计算量并提升特征鲁棒性，前两个卷积块后配置0.1概率的Dropout2d层，全连接层模块前配置0.3概率的Dropout层以抑制过拟合。卷积块之后采用自适应平均池化，将特征图压缩为1×1大小并保留128个通道特征，随后连接含128个神经元的线性层与ReLU激活函数，最终通过线性层输出类别结果。模型总参数与可训练参数均为499914，整体以深层卷积堆叠为核心，配合平滑激活与正则化模块构建高效特征提取架构。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第41个epoch时因早停机制终止，训练过程呈现高效且稳定的收敛趋势。训练损失从初始的1.7527持续下降至最终的0.1488，下降幅度前期显著、后期平缓，未出现停滞或反弹；训练准确率从29.99%稳步提升至94.99%，增速均匀且后期未出现饱和，表明模型对训练数据的拟合能力极强。验证损失从1.4589下降至0.6041，验证准确率从43.28%提升至84.64%，训练损失与验证损失的差距始终保持在合理范围，训练准确率与验证准确率的差异也相对稳定，未出现明显过拟合现象。最终测试结果显示，测试损失为0.6163，测试准确率达到84.52%，测试性能与验证性能高度接近，充分证明模型具有优异的泛化能力。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上与采用三个卷积块（每块3个卷积层）的ConvNet一致，核心差异仅在于将激活函数从ReLU替换为GELU，其他如参数规模、卷积配置、优化器（AdamW）等均保持不变。与同结构采用ReLU+AdamW的模型相比，该模型测试准确率从83.01%提升至84.52%，训练准确率从92.08%提升至94.99%，早停epoch从42提前至41，收敛速度与最终性能均有改善；与同结构采用ReLU+Adam的模型相比，测试准确率从84.12%小幅提升至84.52%，泛化能力进一步优化；与结构更简单的模型（MLP最高57.5%、两卷积块60.88%、三卷积块单卷积层71.98%）相比，该模型仍保持显著性能优势，激活函数的优化未掩盖深层卷积结构的核心价值。**

&emsp;&emsp;**产生这种性能提升的核心原因在于GELU与ReLU激活函数的特性差异。ReLU作为硬阈值激活函数，会将所有负输入直接置为0，可能导致部分梯度信息丢失，尤其在深层网络中易加剧梯度消失问题；而GELU是基于高斯误差分布的平滑非线性激活函数，其输出随输入连续变化，对小幅度输入更敏感，能保留更多细粒度的梯度信息，避免因硬阈值导致的信息损耗。在该模型的深层卷积结构中，GELU的平滑激活特性使各层特征传递更连贯，有助于模型学习到更精细的图像层级特征，同时缓解了深层网络的梯度消失风险，提升了训练收敛效率。配合AdamW的独立权重衰减机制、BatchNorm的分布稳定作用以及Dropout的过拟合抑制，GELU进一步释放了深层卷积结构的特征提取潜力，最终实现测试准确率的提升与泛化能力的优化。**

### 7.3.6、增大卷集核尺寸

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 32,
            "num_conv_layers": 3,
            "kernel_size": 7,
            "pool_size": 2,
        },
        {
            "out_channels": 64,
            "num_conv_layers": 3,
            "kernel_size": 7,
            "pool_size": 2,
        },
        {
            "out_channels": 128,
            "num_conv_layers": 3,
            "kernel_size": 7,
            "pool_size": 2,
        },
    ],
    "activation": "gelu",
    "fc_layers_config": [128],
    "dropout_rates": [0.1, 0.1, 0.3],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型适配CIFAR-10图像分类任务，输入通道数为3，对应RGB图像，输入尺寸为32×32，输出10个类别以匹配数据集类别数。模型包含三个卷积块，每个卷积块内部均堆叠3个7×7卷积层，第一个卷积块输出通道为32，第二个为64，第三个为128，每个卷积层后均跟随BatchNorm2d层和GELU激活函数，以稳定训练数据分布并实现平滑的非线性特征转换。每个卷积块末尾设置2×2最大池化层，用于降低特征图尺寸、减少计算量并提升特征对图像位移的鲁棒性，前两个卷积块后配置0.1概率的Dropout2d层，全连接层模块前配置0.3概率的Dropout层以抑制过拟合。卷积块之后采用自适应平均池化操作，将最终特征图压缩为1×1大小并保留128个通道的特征信息，随后连接含128个神经元的线性层与GELU激活函数，最终通过线性层输出类别预测结果。模型总参数与可训练参数均为2633674，显著多于此前同结构采用3×3卷积核的模型，整体以大感受野卷积堆叠为核心，构建了更大容量的特征提取架构。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第39个epoch时因早停机制终止，训练过程呈现出“快速拟合但过拟合风险加剧”的特点。训练损失从初始的1.7386持续快速下降至最终的0.0877，下降幅度显著且全程无停滞；训练准确率从31.54%稳步提升至97.17%，后期甚至接近饱和，表明模型对训练数据的拟合能力极强。但验证指标呈现明显的“先降后升”趋势，验证损失从1.5104降至低谷后逐步上升至0.8306，验证准确率从41.58%提升至峰值后缓慢回落至82.96%，训练损失与验证损失的差距随epoch增加不断扩大，训练准确率与验证准确率的差异也从初期的不足10%扩大至后期的近14%，明显出现过拟合现象。最终测试结果显示，测试损失为0.8613，测试准确率为82.63%，测试性能不仅低于训练性能，也低于同结构采用3×3卷积核的模型，进一步验证了过拟合对泛化能力的负面影响。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上的核心差异是将卷积核尺寸从3×3扩大至7×7，其他配置如卷积块数量（3个）、每个块的卷积层数（3层）、激活函数（GELU）、优化器（AdamW）、全连接层结构（128神经元）等均保持一致，但参数规模从499914大幅增至2633674，模型容量显著提升。从训练表现来看，该模型的训练准确率（97.17%）高于此前所有模型，包括同结构3×3卷积核的模型（94.99%），但验证准确率（82.96%）和测试准确率（82.63%）均低于后者（验证84.64%、测试84.52%），早停epoch也从41提前至39，过拟合现象更为严重。与结构更简单的模型（如MLP、两卷积块ConvNet、三卷积块单卷积层ConvNet）相比，该模型的测试准确率仍处于较高水平，但性能优势已明显缩小，不再像3×3卷积核模型那样具有显著突破。**

&emsp;&emsp;**产生这种变化的核心原因在于卷积核尺寸扩大对模型的双重影响。一方面，7×7卷积核的感受野远大于3×3，能一次性捕捉图像中更大范围的像素关联信息，理论上更适合提取全局结构特征，这也是该模型训练准确率更高的重要原因；另一方面，卷积核尺寸扩大导致参数数量急剧增加（7×7卷积核的参数数量是3×3的5倍以上），模型容量远超CIFAR-10小图像数据集的需求，即使配置了Dropout层和AdamW的权重衰减机制，也难以完全抑制过拟合风险。同时，CIFAR-10图像尺寸仅为32×32，7×7卷积核的感受野已接近图像一半大小，过大的感受野反而可能捕捉到冗余的背景信息或噪声，而非有效的分类特征，进一步加剧了过拟合。此外，参数数量增加还导致模型对训练数据的“记忆效应”增强，虽然能更好地拟合训练样本，但对未见过的测试样本适应性下降，最终造成验证和测试性能的回落，使得卷积核扩大带来的感受野优势被过拟合风险完全抵消，甚至出现性能倒退。**

### 7.3.7、更多优化的网络结构

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.15, 0.25, 0.35, 0.4],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型专为CIFAR-10图像分类任务设计，输入通道数为3，以适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含三个卷积块，第一个卷积块输出通道96，内含2个3×3卷积层，第二个输出通道192、第三个384，每块均保持2个3×3卷积层的配置；每个卷积层后均紧跟BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以加速收敛，后者通过平滑非线性转换保留细粒度梯度信息。每个卷积块末尾设置2×2最大池化层，降低特征图尺寸的同时提升特征鲁棒性，且卷积块对应的Dropout2d率按0.15、0.25递增，针对深层特征过拟合风险动态调整抑制力度。卷积块后采用自适应平均池化，将特征图压缩为1×1大小并保留384个通道特征，随后连接含384个神经元的全连接层，配合0.4概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果。模型总参数与可训练参数均为2729578，通过高通道数、分层正则化的设计构建了强特征表达能力的架构。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第41个epoch时因早停机制终止，整体呈现“高效收敛且过拟合控制优异”的表现。训练损失从初始1.5297持续快速下降至0.0755，下降过程无停滞，且后期降幅平缓，表明模型对训练数据的拟合能力极强且未出现无效迭代；训练准确率从41.98%稳步提升至97.38%，增速前期快、后期趋于饱和，反映模型学习过程稳定且充分。验证损失从1.1579降至0.4218，验证准确率从57.54%提升至89.20%，训练损失与验证损失的差距始终控制在0.3以内，训练准确率与验证准确率的差异也稳定在8%-9%，未出现随epoch扩大的趋势，过拟合现象被有效抑制。最终测试结果显示，测试损失0.4331、测试准确率89.43%，测试性能与验证性能高度接近，甚至测试准确率略高于验证准确率，充分证明模型具有出色的泛化能力。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构与性能上均实现显著突破。结构方面，此前模型如3个卷积块（每块3层3×3核）的参数仅499914、通道数32/64/128，7×7核模型虽参数达2633674但通道数相同，而该模型将通道数提升至96/192/384，同时将每块卷积层数从3层减至2层，在提升特征表达能力的同时平衡参数效率，且Dropout率按层递增的设计更具针对性。训练表现上，该模型测试准确率89.43%远超所有前期模型，包括3×3核高卷积层数模型（84.52%）、7×7核模型（82.63%），以及结构更简单的MLP（最高57.5%）、两卷积块（60.88%）、三卷积块单卷积层（71.98%）模型；同时，其验证与测试性能的一致性优于7×7核模型，过拟合控制效果更优，收敛速度也快于多数模型，前10个epoch训练准确率即可从41.98%提升至81.76%。**

&emsp;&emsp;**产生这种性能飞跃的核心原因在于结构设计的多维度优化。首先，高通道数（96/192/384）让每个卷积块能提取更丰富的特征维度，相比低通道数模型可捕捉更多图像细节信息，且每块2个卷积层的配置在保证特征提取深度的同时，避免了3层卷积可能导致的参数冗余与过拟合风险。其次，分层递增的Dropout率（0.15→0.25→0.4）精准匹配不同层的过拟合风险，浅层特征过拟合概率低故用低Dropout率保留特征，深层与全连接层过拟合风险高则用高Dropout率强力抑制，比固定Dropout率更高效。再者，GELU激活的平滑特性与高通道数结合，进一步释放了特征传递的连贯性，配合AdamW的独立权重衰减、BatchNorm的分布稳定作用，有效缓解了高参数规模下的过拟合风险。最后，自适应平均池化与384神经元全连接层的尺寸匹配，确保卷积提取的高维特征能被全连接层充分利用，避免特征维度浪费，最终实现特征表达能力与泛化能力的双重提升。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 768,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.1, 0.2, 0.3, 0.4, 0.5],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型用于CIFAR-10图像分类任务，输入通道数为3，对应RGB图像，输入尺寸32×32，输出10个类别以匹配数据集类别数。模型包含四个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次递增为96、192、384、768，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，用于降低特征图尺寸、减少计算量并提升特征鲁棒性，且卷积块与全连接层对应不同的Dropout率，按0.1、0.2、0.3、0.4、0.5递增，针对深层与全连接层更高的过拟合风险动态调整抑制力度。卷积块之后采用自适应平均池化，将最终特征图压缩为1×1大小并保留768个通道特征，随后连接含384个神经元的全连接层，配合0.5概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果。模型总参数与可训练参数均为10844266，通过增加卷积块数量与通道数，构建了远超此前同类模型的大容量特征提取架构。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第32个epoch时因早停机制终止，训练表现呈现“强拟合能力但过拟合风险显著加剧”的特点。训练损失从初始的1.4278持续快速下降至0.0327，下降幅度贯穿全程且无停滞；训练准确率从47.09%稳步提升至98.96%，后期接近饱和，表明模型对训练数据的拟合能力极强。但验证指标呈现“先优化后恶化”的趋势，验证损失从1.0358降至低谷后逐步上升至0.6204，验证准确率从62.48%提升至峰值后回落至88.42%，训练损失与验证损失的差距从初期的0.39扩大至后期的0.59，训练准确率与验证准确率的差异更是从3%扩大至10%以上，过拟合现象明显。最终测试结果显示，测试损失为0.6488，测试准确率为88.03%，不仅低于训练性能，也低于此前同系列3个卷积块模型的测试准确率，且测试损失高于验证损失，进一步验证过拟合对泛化能力的负面影响。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上的核心变化是增加1个输出通道为768的卷积块，使卷积块总数从3个增至4个，参数规模从2729578大幅增至10844266，其他配置如每个块的卷积层数（2层）、激活函数（GELU）、优化器（AdamW）、全连接层结构（384神经元）、Dropout递增策略等均与3个卷积块的最优模型一致。从训练表现来看，该模型的训练准确率（98.96%）是所有模型中最高的，远超3个卷积块模型的97.38%，但验证准确率（88.42%）和测试准确率（88.03%）均低于后者（验证89.20%、测试89.43%），早停epoch也从41提前至32，过拟合程度更严重。与结构更简单的模型（如7×7核模型、三卷积块单卷积层模型、MLP等）相比，该模型的测试准确率仍处于较高水平，但性能优势已明显缩小，不再具备3个卷积块模型的显著领先地位。**

&emsp;&emsp;**产生这种性能回落的核心原因在于模型容量与数据集规模的不匹配。CIFAR-10作为小样本图像数据集，样本数量有限且特征复杂度较低，而该模型通过增加卷积块数量与768通道数，使参数规模突破千万，远超数据集所能承载的特征学习需求，形成“模型过配”。尽管模型配置了递增Dropout率与AdamW权重衰减，但面对千万级参数的过拟合风险，这些正则化措施的抑制效果已达上限，无法阻止模型对训练数据中噪声与冗余信息的“记忆”。同时，第四个768通道的卷积块虽进一步提升特征维度，但在CIFAR-10的简单特征场景下，反而导致特征冗余，增加了模型学习有效分类特征的难度，甚至出现“特征混淆”。此外，更深的卷积结构还可能加剧梯度传递过程中的微小误差累积，虽有BatchNorm与GELU缓解，但深层特征的优化效率仍低于浅层，最终导致验证与测试性能回落，出现“训练性能优异但泛化能力下降”的矛盾现象。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 768,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.2, 0.3, 0.4, 0.5, 0.6],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型针对CIFAR-10图像分类任务设计，输入通道数为3，以适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含四个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384、768，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失问题，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，用于降低特征图尺寸、减少计算量并提升特征对图像位移的鲁棒性，同时Dropout率按卷积块与全连接层依次递增为0.2、0.3、0.4、0.5、0.6，针对深层与全连接层更高的过拟合风险强化抑制效果。卷积块之后采用自适应平均池化，将最终特征图压缩为1×1大小并保留768个通道特征，随后连接含384个神经元的全连接层，配合0.6概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果。模型总参数与可训练参数均为10844266，保持了四卷积块架构的大容量特征提取能力，同时通过调整Dropout率增强正则化。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第36个epoch时因早停机制终止，整体呈现“拟合能力强且过拟合风险有效缓解”的表现。训练损失从初始的1.5487持续下降至0.0577，下降过程平稳且无停滞；训练准确率从42.28%稳步提升至98.07%，后期增速放缓但未出现异常波动，表明模型对训练数据的拟合能力依然强劲。验证指标表现优于此前同架构模型，验证损失从1.1056降至低谷后虽有波动，但最终稳定在0.5126左右，验证准确率从59.64%提升至88.98%，训练损失与验证损失的差距控制在0.5以内，训练准确率与验证准确率的差异也从初期的20%缩小至后期的10%左右，过拟合现象较此前四卷积块模型明显减轻。最终测试结果显示，测试损失为0.5486，测试准确率为88.45%，测试性能与验证性能高度接近，泛化能力得到显著改善。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上与此前四卷积块模型（通道96/192/384/768、2层3×3卷积）一致，核心差异在于两点：一是Dropout率整体提升，各层Dropout率从0.1/0.2/0.3/0.4/0.5增至0.2/0.3/0.4/0.5/0.6；二是学习率调度器的step_size从20调整为10，学习率下降频率更高。从训练表现来看，与此前四卷积块模型相比，该模型的测试准确率从88.03%提升至88.45%，验证损失波动幅度减小，过拟合程度显著减轻，训练准确率虽从98.96%降至98.07%，但拟合效率未受明显影响；与三卷积块最优模型（测试准确率89.43%）相比，仍存在小幅差距，但性能差距从1.4个百分点缩小至0.98个百分点；与结构更简单的模型（如7×7核模型、三卷积块单卷积层模型）相比，该模型的测试准确率仍保持明显优势，进一步巩固了大容量架构的性能基础。**

&emsp;&emsp;**产生这种性能改善的核心原因在于正则化策略与学习率调度的优化调整。首先，Dropout率的整体提升增强了对过拟合的抑制力度，尤其是深层卷积块与全连接层的Dropout率提高，有效减少了模型对训练数据中噪声与冗余信息的“记忆”，同时避免了过度抑制导致的特征丢失，在拟合能力与泛化能力间取得更好平衡。其次，学习率调度器step_size从20调整为10，使学习率每10个epoch下降一次，更频繁的学习率衰减有助于模型在训练中后期快速稳定参数，避免因学习率过高导致的参数震荡，减少了验证指标的波动，进一步提升了泛化稳定性。此外，BatchNorm2d与GELU激活函数的协同作用，持续保障了深层特征传递的连贯性，缓解了大容量模型的梯度消失风险，为正则化策略的生效提供了基础。尽管模型容量仍远超CIFAR-10数据集需求，但通过上述优化，过拟合风险得到有效控制，最终实现了测试准确率的提升与泛化能力的改善。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 768,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.1, 0.2, 0.4, 0.5, 0.6],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型针对CIFAR-10图像分类任务构建，输入通道数为3，以适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含四个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384、768，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，用于降低特征图尺寸、减少计算量并提升特征鲁棒性，Dropout率则按卷积块与全连接层依次为0.1、0.2、0.4、0.5、0.6，其中第三卷积块Dropout率较此前同类模型有所提升，针对深层特征过拟合风险强化抑制。卷积块后采用自适应平均池化，将特征图压缩为1×1大小并保留768个通道特征，随后连接含384个神经元的全连接层，配合0.6概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果，模型总参数与可训练参数均为10844266，保持四卷积块架构的大容量特征提取能力。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第31个epoch时因早停机制终止，训练表现呈现“强拟合能力但验证稳定性下降”的特点。训练损失从初始的1.4651持续快速下降至0.0598，下降过程无停滞；训练准确率从45.52%稳步提升至98.04%，后期接近饱和，表明模型对训练数据的拟合能力依然强劲。但验证指标波动幅度加大，验证损失从1.0840降至低谷后逐步上升至0.5894，验证准确率从60.94%提升至峰值后回落至87.34%，训练损失与验证损失的差距从初期的0.38扩大至后期的0.53，训练准确率与验证准确率的差异也从15%扩大至30%以上，过拟合迹象较此前优化过的四卷积块模型更明显。最终测试结果显示，测试损失为0.5967，测试准确率为87.82%，虽与验证性能基本匹配，但低于此前采用StepLR调度的四卷积块模型，泛化稳定性有所下降。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上的核心变化有两点：一是第三卷积块的Dropout率从0.3提升至0.4，强化对深层特征的过拟合抑制；二是学习率调度器从StepLR（step_size=10、gamma=0.5）更换为CosineAnnealingLR（T_max=100），学习率更新策略从固定步长下降变为余弦周期波动。从训练表现来看，与此前采用StepLR的四卷积块模型相比，该模型的测试准确率从88.45%降至87.82%，验证损失波动幅度更大，早停epoch从36提前至31，过拟合程度稍重；与三卷积块最优模型（测试准确率89.43%）相比，仍存在1.6个百分点的差距；与结构更简单的模型（如7×7核模型、三卷积块单卷积层模型）相比，测试准确率仍处于较高水平，但性能优势进一步缩小，大容量架构的潜力未充分发挥。**

&emsp;&emsp;**产生这种性能变化的核心原因在于学习率调度策略的差异。CosineAnnealingLR通过余弦函数周期性调整学习率，在训练后期会出现学习率回升，这种波动虽可能帮助模型跳出局部最优，但对于CIFAR-10这类小样本数据集，反而容易导致参数更新不稳定，尤其是在模型容量已过大的情况下，后期学习率回升可能加剧对训练数据噪声的“记忆”，进而导致验证损失上升与准确率回落。而此前的StepLR采用固定步长下降，学习率逐步降低并保持稳定，更适合小数据集后期的参数微调，能有效抑制过拟合。此外，第三卷积块Dropout率提升至0.4，虽强化了深层特征的正则化，但对整体泛化能力的改善效果被学习率调度的负面影响抵消，加上模型千万级参数的容量仍远超CIFAR-10的需求，过拟合风险本就较高，最终导致测试准确率低于采用StepLR调度的同类模型。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.15, 0.25, 0.35, 0.4],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型专为CIFAR-10图像分类任务设计，输入通道数为3，以适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含三个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失问题，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，用于降低特征图尺寸、减少计算量并提升特征对图像位移的鲁棒性，Dropout率则按卷积块与全连接层依次递增为0.15、0.25、0.35、0.4，针对深层与全连接层更高的过拟合风险动态调整抑制力度。卷积块之后采用自适应平均池化，将最终特征图压缩为1×1大小并保留384个通道特征，随后连接含384个神经元的全连接层，配合0.4概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果，模型总参数与可训练参数均为2729578，通过合理的卷积块数量与通道配置，构建了兼顾特征表达能力与参数效率的架构。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第42个epoch时因早停机制终止，整体呈现“高效收敛且过拟合控制优异”的表现。训练损失从初始的1.5407持续平稳下降至0.0623，下降过程无停滞且后期降幅平缓，表明模型对训练数据的拟合能力强劲且未出现无效迭代；训练准确率从41.54%稳步提升至97.77%，增速前期快、后期趋于饱和，反映模型学习过程稳定且充分。验证指标同样保持良好趋势，验证损失从1.1445降至0.4498，验证准确率从58.78%提升至89.42%，训练损失与验证损失的差距始终控制在0.4以内，训练准确率与验证准确率的差异也稳定在8%-10%，未出现随epoch扩大的趋势，过拟合现象被有效抑制。最终测试结果显示，测试损失为0.4634，测试准确率为89.03%，测试性能与验证性能高度接近，充分证明模型具有出色的泛化能力。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上的核心变化是减少1个输出通道为768的卷积块，使卷积块总数从4个减至3个，参数规模从10844266大幅降至2729578，同时Dropout率调整为更平缓的递增梯度（0.15→0.25→0.35→0.4），学习率调度器采用CosineAnnealingLR（T_max=100）。从训练表现来看，与四卷积块模型（测试准确率87.82%）相比，该模型测试准确率提升1.21个百分点，验证损失波动更小，过拟合程度显著减轻；与此前采用StepLR调度的三卷积块最优模型（测试准确率89.43%）相比，虽存在0.4个百分点的小幅差距，但在使用CosineAnnealingLR的情况下仍保持接近性能，泛化稳定性更优；与结构更简单的模型（如MLP最高57.5%、7×7核模型82.63%、三卷积块单卷积层模型71.98%）相比，该模型的测试准确率仍保持显著优势，进一步验证了三卷积块架构的合理性。**

&emsp;&emsp;**产生这种优异表现的核心原因在于模型容量与数据集需求的精准匹配。CIFAR-10作为小样本图像数据集，无需四卷积块千万级参数的超大容量，三卷积块配合96/192/384通道的配置，既能提供足够的特征表达能力以捕捉图像多层级特征，又避免了参数冗余导致的过拟合风险。同时，CosineAnnealingLR的周期性学习率调整虽在理论上存在波动，但因模型容量适中，参数更新不易受学习率波动影响，反而能在训练后期帮助模型跳出局部最优，提升泛化潜力；而平缓递增的Dropout率则精准匹配各层过拟合风险，既保留浅层有效特征，又强力抑制深层冗余信息，避免过度正则化导致的性能损失。此外，GELU激活的平滑特性、BatchNorm的分布稳定作用与AdamW的独立权重衰减协同工作，进一步保障了特征传递的连贯性与参数优化的稳定性，最终实现拟合能力与泛化能力的平衡，达到接近三卷积块最优模型的性能水平。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.1, 0.2, 0.3, 0.4],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型针对CIFAR-10图像分类任务构建，输入通道数为3，适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含三个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，降低特征图尺寸、减少计算量的同时提升特征鲁棒性，Dropout率则按卷积块与全连接层依次递增为0.1、0.2、0.3、0.4，针对不同层过拟合风险梯度调整抑制力度。卷积块后采用自适应平均池化，将特征图压缩为1×1大小并保留384个通道特征，随后连接含384个神经元的全连接层，配合0.4概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果，模型总参数与可训练参数均为2729578，兼顾特征表达能力与参数效率。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第38个epoch时因早停机制终止，整体呈现“拟合能力强劲但泛化稳定性略有下降”的表现。训练损失从初始1.4759持续快速下降至0.0527，下降过程无停滞，后期降幅平缓；训练准确率从44.29%稳步提升至98.21%，后期接近饱和，表明模型对训练数据的拟合能力充分。但验证指标波动幅度较此前同类优化模型略大，验证损失从1.1149降至低谷后回升至0.5174，验证准确率从59.18%提升至峰值后回落至88.42%，训练损失与验证损失的差距从初期0.36扩大至后期0.46，训练准确率与验证准确率的差异也从15%扩大至29%，过拟合迹象较此前优化的三卷积块模型稍明显。最终测试结果显示，测试损失0.5174、测试准确率88.44%，测试性能与验证性能基本匹配，但泛化能力较此前部分三卷积块模型有所回落。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上与此前三卷积块模型（通道96/192/384、2层3×3卷积）一致，核心差异仅在于Dropout率调整，将浅层卷积块的Dropout率从0.15降至0.1，整体梯度变为0.1→0.2→0.3→0.4，学习率调度器仍为CosineAnnealingLR（T_max=100）。从训练表现来看，与此前Dropout率0.15→0.25→0.35→0.4的三卷积块模型（测试准确率89.03%）相比，该模型测试准确率下降0.59个百分点，验证损失波动更大，过拟合程度略重；与四卷积块模型（测试准确率87.82%）相比，仍保持0.62个百分点的优势；与采用StepLR的三卷积块最优模型（测试准确率89.43%）相比，差距扩大至1个百分点；与结构更简单的MLP、7×7核模型等相比，仍保持显著性能优势，但优势幅度有所缩小。**

&emsp;&emsp;**产生这种性能变化的核心原因在于浅层Dropout率的调整。此前三卷积块模型将浅层Dropout率设为0.15，能适度抑制浅层特征提取过程中的噪声冗余，而该模型将其降至0.1，对浅层过拟合的抑制力度减弱，导致浅层保留了更多训练数据中的噪声信息。这些噪声随特征传递至深层，虽未严重影响模型对训练数据的拟合，但在面对未见过的测试数据时，会干扰有效特征的判断，进而导致泛化能力下降。同时，CosineAnnealingLR的周期性学习率波动虽能帮助模型跳出局部最优，但在浅层噪声增多的情况下，这种波动未能完全抵消噪声带来的负面影响，反而轻微加剧了验证指标的波动。不过，由于模型仍保持三卷积块的合理容量，未出现四卷积块的参数冗余问题，因此测试准确率仍高于四卷积块模型，只是较此前优化的三卷积块模型略有回落。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.2, 0.3, 0.4, 0.5],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型针对CIFAR-10图像分类任务设计，输入通道数为3，适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含三个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失问题，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，降低特征图尺寸、减少计算量的同时提升特征对图像位移的鲁棒性，Dropout率则按卷积块与全连接层依次递增为0.2、0.3、0.4、0.5，针对浅层至深层过拟合风险逐步提升抑制力度。卷积块后采用自适应平均池化，将特征图压缩为1×1大小并保留384个通道特征，随后连接含384个神经元的全连接层，配合0.5概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果，模型总参数与可训练参数均为2729578，保持三卷积块架构的合理容量与特征表达能力。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第49个epoch时因早停机制终止，整体呈现“收敛稳定且过拟合控制优异”的表现。训练损失从初始1.5970持续平稳下降至0.0660，下降过程无停滞且后期降幅平缓，表明模型对训练数据的拟合能力强劲且未出现无效迭代；训练准确率从38.91%稳步提升至97.80%，增速前期快、后期趋于饱和，反映模型学习过程充分且稳定。验证指标同样表现出色，验证损失从1.2196降至0.4463，验证准确率从54.46%提升至89.42%，训练损失与验证损失的差距始终控制在0.4以内，训练准确率与验证准确率的差异也稳定在8%-10%，未出现随epoch扩大的趋势，过拟合现象被有效抑制。最终测试结果显示，测试损失0.4506、测试准确率88.92%，测试性能与验证性能高度接近，泛化能力显著优于多数同类模型。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上的核心变化是整体提升Dropout率，将浅层卷积块Dropout率从0.1或0.15提升至0.2，深层与全连接层同步调整为0.3、0.4、0.5，学习率调度器仍为CosineAnnealingLR（T_max=100）。从训练表现来看，与此前Dropout率0.1→0.2→0.3→0.4的三卷积块模型（测试准确率88.44%）相比，该模型测试准确率提升0.48个百分点，验证损失波动更小，过拟合控制更优；与四卷积块模型（测试准确率87.82%）相比，保持1.1个百分点的优势；与采用StepLR的三卷积块最优模型（测试准确率89.43%）相比，差距缩小至0.51个百分点；与结构更简单的MLP、7×7核模型、三卷积块单卷积层模型相比，测试准确率仍保持显著优势，进一步巩固了三卷积块架构的性能优势。**

&emsp;&emsp;**产生这种性能提升的核心原因在于Dropout率的合理上调。此前部分模型因浅层Dropout率偏低（0.1或0.15），未能充分抑制浅层特征提取过程中的噪声冗余，导致少量噪声随特征传递至深层，影响泛化能力；该模型将浅层Dropout率提升至0.2，在不损失有效特征的前提下，更精准地过滤了浅层冗余信息，减少了噪声对深层特征学习的干扰。同时，深层与全连接层Dropout率同步提升，进一步强化对高过拟合风险层的抑制，形成“分层递进”的正则化效果，避免了单一Dropout率难以适配不同层风险的问题。配合CosineAnnealingLR的周期性学习率调整，模型既能在训练前期快速收敛，又能在后期通过学习率波动跳出局部最优，加上GELU激活的平滑特性与BatchNorm的分布稳定作用，最终实现拟合能力与泛化能力的平衡，使测试准确率接近三卷积块模型的最优水平。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.2, 0.3, 0.5, 0.6],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型针对CIFAR-10图像分类任务构建，输入通道数为3，适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含三个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，降低特征图尺寸、减少计算量的同时提升特征鲁棒性，Dropout率则按卷积块与全连接层依次递增为0.2、0.3、0.5、0.6，其中第三卷积块与全连接层的Dropout率较此前同类模型进一步提升，强化对高过拟合风险层的抑制。卷积块后采用自适应平均池化，将特征图压缩为1×1大小并保留384个通道特征，随后连接含384个神经元的全连接层，配合0.6概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果，模型总参数与可训练参数均为2729578，保持三卷积块架构的合理容量。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第46个epoch时因早停机制终止，整体呈现“收敛高效且过拟合控制稳定”的表现。训练损失从初始1.6308持续平稳下降至0.0915，下降过程无停滞且后期降幅平缓，表明模型对训练数据的拟合能力强劲且未出现无效迭代；训练准确率从37.27%稳步提升至96.84%，增速前期快、后期趋于饱和，反映模型学习过程充分且稳定。验证指标同样表现良好，验证损失从1.2588降至低谷后小幅波动，最终稳定在0.4816左右，验证准确率从52.06%提升至88.56%，训练损失与验证损失的差距始终控制在0.4以内，训练准确率与验证准确率的差异也稳定在8%-10%，未出现随epoch扩大的趋势，过拟合现象被有效抑制。最终测试结果显示，测试损失0.4379、测试准确率89.03%，测试性能与验证性能高度接近，泛化能力达到三卷积块模型的较高水平。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上的核心变化是提升第三卷积块与全连接层的Dropout率，从此前的0.4、0.5分别上调至0.5、0.6，其他配置如卷积块数量、通道数、激活函数（GELU）、学习率调度器（CosineAnnealingLR）等均保持一致。从训练表现来看，与上一个Dropout率为0.2→0.3→0.4→0.5的三卷积块模型（测试准确率88.92%）相比，该模型测试准确率提升0.11个百分点，验证损失波动更小，过拟合控制更优；与四卷积块模型（测试准确率87.82%）相比，保持1.21个百分点的优势；与采用StepLR的三卷积块最优模型（测试准确率89.43%）相比，差距进一步缩小至0.4个百分点；与结构更简单的MLP、7×7核模型、三卷积块单卷积层模型相比，测试准确率仍保持显著优势，进一步验证了三卷积块架构配合精准Dropout调整的有效性。**

&emsp;&emsp;**产生这种性能提升的核心原因在于对高风险层Dropout率的精准上调。第三卷积块作为深层特征提取层，易积累浅层传递的冗余信息，全连接层则直接处理高维特征，二者均为过拟合高发层，此前0.4、0.5的Dropout率虽能抑制过拟合，但仍存在少量冗余残留；将其分别提升至0.5、0.6后，在不损失有效特征的前提下，更彻底地过滤了深层冗余与噪声，减少了模型对训练数据“记忆”的可能性。同时，这种调整与GELU激活的平滑特性、BatchNorm的分布稳定作用形成协同，确保深层特征传递的连贯性不受高Dropout率影响；配合CosineAnnealingLR的周期性学习率调整，模型在训练后期仍能稳定优化参数，避免因过拟合导致的性能回落。最终，通过对高风险层的针对性正则化优化，模型实现了拟合能力与泛化能力的更优平衡，测试准确率进一步接近三卷积块模型的最优水平。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 768,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.15, 0.25, 0.35, 0.6, 0.7],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 1e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型针对CIFAR-10图像分类任务构建，输入通道数为3，适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含四个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384、768，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，降低特征图尺寸、减少计算量的同时提升特征鲁棒性，Dropout率则按卷积块与全连接层依次递增为0.15、0.25、0.35、0.6、0.7，其中第四卷积块与全连接层的Dropout率大幅提升，针对深层高过拟合风险层强化抑制。卷积块后采用自适应平均池化，将特征图压缩为1×1大小并保留768个通道特征，随后连接含384个神经元的全连接层，配合0.7概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果，模型总参数与可训练参数均为10844266，保持四卷积块架构的大容量特征提取能力。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第38个epoch时因早停机制终止，整体呈现“收敛高效且过拟合风险显著缓解”的表现。训练损失从初始1.5474持续快速下降至0.0529，下降过程无停滞且后期降幅平缓，表明模型对训练数据的拟合能力强劲；训练准确率从37.27%稳步提升至98.25%，增速前期快、后期趋于饱和，反映模型学习过程充分。验证指标较此前四卷积块模型明显改善，验证损失从1.0760降至低谷后小幅波动，最终稳定在0.5673左右，验证准确率从52.06%提升至88.90%，训练损失与验证损失的差距控制在0.5以内，训练准确率与验证准确率的差异也稳定在10%左右，过拟合现象较此前四卷积块模型大幅减轻。最终测试结果显示，测试损失0.5610、测试准确率88.52%，测试性能与验证性能高度接近，泛化能力较此前四卷积块模型显著提升。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构上的核心变化有两点：一是保留四卷积块架构（通道96/192/384/768），参数规模维持10844266；二是大幅提升深层与全连接层的Dropout率，将第四卷积块从0.5上调至0.6、全连接层从0.6上调至0.7，其他配置如激活函数（GELU）、学习率调度器（CosineAnnealingLR）等保持一致。从训练表现来看，与此前Dropout率较低的四卷积块模型（测试准确率87.82%）相比，该模型测试准确率提升0.7个百分点，验证损失波动更小，过拟合控制更优；与三卷积块最优模型（测试准确率89.43%）相比，仍存在0.91个百分点的差距；与结构更简单的MLP、7×7核模型相比，测试准确率仍保持显著优势，但优势幅度小于三卷积块模型，进一步验证四卷积块架构需依赖强正则化才能接近三卷积块的泛化能力。**

&emsp;&emsp;**产生这种性能变化的核心原因在于深层Dropout率的大幅上调。此前四卷积块模型因深层与全连接层Dropout率不足，难以抑制千万级参数的过拟合风险；该模型将第四卷积块与全连接层Dropout率分别提升至0.6、0.7，更彻底地过滤了深层冗余特征与噪声，减少了模型对训练数据“记忆”的可能性，有效缓解了过拟合。同时，GELU激活的平滑特性与BatchNorm的分布稳定作用，确保高Dropout率下深层特征传递仍保持连贯，避免因特征过度丢弃导致的拟合能力下降；配合CosineAnnealingLR的周期性学习率调整，模型在训练后期仍能稳定优化参数，进一步提升泛化稳定性。但受限于四卷积块的超大容量（1084万参数）仍远超CIFAR-10数据集需求，即使强化正则化，仍无法完全抵消参数冗余的影响，因此测试准确率仍低于三卷积块最优模型，仅能接近其性能水平。**

In [ ]:
# 模型配置
model_config = {
    "conv_layers_config": [
        {
            "out_channels": 96,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 192,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 384,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
        {
            "out_channels": 768,
            "num_conv_layers": 2,
            "kernel_size": 3,
            "pool_size": 2,
        },
    ],
    "fc_layers_config": [384],
    "activation": "gelu",
    "global_pool": "adaptive_avg",
    "dropout_rates": [0.2, 0.3, 0.4, 0.6, 0.7],
}

# 超参数配置
config = {
    "batch_size": 100,
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "weight_decay": 5e-4,
}

# 训练ConvNet模型
print("=" * 28)
print("Training ConvNet Model:")
print("=" * 28)

convnet_model = ConvNet(**model_config)
print("Model Information:", convnet_model.get_model_info())
convnet_trainer = Trainer(convnet_model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    convnet_model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["max_epochs"])

convnet_results = convnet_trainer.train(
    train_loader, val_loader, criterion, optimizer, scheduler, config["max_epochs"]
)

_, convnet_test_acc = convnet_trainer.test(test_loader, criterion)

print("\nConvNet Training Curves:")
plot_training_curves(
    convnet_results["train_losses"],
    convnet_results["val_losses"],
    convnet_results["train_accs"],
    convnet_results["val_accs"],
)
print(f"ConvNet Test Accuracy: {convnet_test_acc:.2f}%")

&emsp;&emsp;**该ConvNet模型针对CIFAR-10图像分类任务构建，输入通道数为3，适配RGB图像，输入尺寸32×32，输出10个类别对应数据集类别数。模型包含四个卷积块，每个卷积块内部均堆叠2个3×3卷积层，输出通道数按块依次为96、192、384、768，每个卷积层后均跟随BatchNorm2d层与GELU激活函数，前者稳定各层输入数据分布以缓解梯度消失，后者通过平滑非线性转换保留细粒度特征信息。每个卷积块末尾设置2×2最大池化层，降低特征图尺寸、减少计算量的同时提升特征鲁棒性，Dropout率则按卷积块与全连接层依次递增为0.2、0.3、0.4、0.6、0.7，针对浅层至深层过拟合风险逐步强化抑制。卷积块后采用自适应平均池化，将特征图压缩为1×1大小并保留768个通道特征，随后连接含384个神经元的全连接层，配合0.7概率的Dropout层与GELU激活函数，最终通过线性层输出类别结果，模型总参数与可训练参数均为10844266，保持四卷积块架构的大容量特征提取能力。**

&emsp;&emsp;**该模型在CIFAR-10数据集上训练至第41个epoch时因早停机制终止，整体呈现“收敛高效且过拟合控制进一步优化”的表现。训练损失从初始1.5957持续平稳下降至0.0552，下降过程无停滞且后期降幅平缓，表明模型对训练数据的拟合能力强劲；训练准确率从39.80%稳步提升至98.19%，增速前期快、后期趋于饱和，反映模型学习过程充分。验证指标较此前四卷积块模型更稳定，验证损失从1.1395降至低谷后小幅波动，最终稳定在0.6039左右，验证准确率从58.12%提升至88.48%，训练损失与验证损失的差距控制在0.55以内，训练准确率与验证准确率的差异也稳定在10%左右，过拟合现象较此前四卷积块模型进一步减轻。最终测试结果显示，测试损失0.5695、测试准确率88.70%，测试性能与验证性能高度接近，泛化能力较此前四卷积块模型显著提升。**

&emsp;&emsp;**与之前讨论的所有模型相比，该模型在结构与超参数上的核心变化有两点：一是保持四卷积块架构（通道96/192/384/768）与Dropout率配置（0.2→0.3→0.4→0.6→0.7）不变，参数规模仍为10844266；二是将权重衰减从1e-4提升至5e-4，强化对参数规模的约束。从训练表现来看，与此前权重衰减1e-4的四卷积块模型（测试准确率88.52%）相比，该模型测试准确率提升0.18个百分点，验证损失波动更小，过拟合控制更优；与三卷积块最优模型（测试准确率89.43%）相比，差距缩小至0.73个百分点；与结构更简单的MLP、7×7核模型相比，测试准确率仍保持显著优势，但优势幅度小于三卷积块模型，进一步验证四卷积块架构需依赖“高Dropout率+高权重衰减”的组合正则化，才能接近三卷积块的泛化能力。**

&emsp;&emsp;**产生这种性能变化的核心原因在于权重衰减与Dropout的协同强化。此前四卷积块模型虽通过高Dropout率缓解过拟合，但权重衰减力度不足，仍有部分参数因缺乏约束而过度拟合训练噪声；该模型将权重衰减提升至5e-4，通过对参数绝对值施加更强惩罚，有效抑制参数过大导致的过拟合风险，与高Dropout率形成“双重保险”——前者从参数规模层面约束，后者从特征选择层面过滤，共同减少模型对训练数据冗余信息的依赖。同时，GELU激活的平滑特性与BatchNorm的分布稳定作用，确保双重正则化下深层特征传递仍保持连贯，避免因参数与特征过度抑制导致的拟合能力下降；配合CosineAnnealingLR的周期性学习率调整，模型在训练后期仍能稳定优化参数，进一步提升泛化稳定性。但受限于四卷积块的超大容量（1084万参数）仍远超CIFAR-10数据集需求，即使强化正则化，仍无法完全抵消参数冗余的影响，因此测试准确率仍低于三卷积块最优模型，仅能进一步缩小性能差距。**